# Measure foci count and nucleus FISH intensities

- scripts to qiantify following parameters
    1. Foci counts per nuclei
    2. Foci intensities per nuclei (total integrated intensities & avg intensities per focus)
    3. Nucleus intensities


## 1. Prepare sum intensity projections

Raw images are (1) split for FISH channel; (2) rolling-ball background subtracted slice-by-slice along the z with user-defiend radius; (3) sum z-projected; and (4) saved with user-defined suffix at a user-defined directory. 

In [1]:
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, "src")

import numpy as np
import tifffile

from imagej_rolling_ball import rolling_ball_background, subtract_background_image

# --- User-defined -------------------------------------------------------------------
INPUT_DIR = "/Volumes/Jeff-exFAT/Marko_HIV/Marko_HIV_rawsorted"
OUTPUT_DIR = "/Volumes/Jeff-exFAT/Marko_HIV/Marko_HIV_ch0bgs100sum"
PATTERN = "*.tif"
CHANNEL_AXIS = 1           # 0-based, in the raw (Z, C, Y, X) order
CHANNEL_INDEX = 0          # 0-based FISH channel
Z_AXIS = 0                 # 0-based Z axis after the channel axis is removed
ROLLING_BALL_RADIUS = 100  # pixels
SUFFIX = "_ch0bgs100sum"
WORKERS = 4                # images processed in parallel; ~1.5 GB RAM each
OVERWRITE = False          # False: skip images whose output already exists


# --- Split channel -> rolling ball per slice -> sum Z-project -> save --------------
def process(path, ref_shape):
    """Returns a list of reasons the file was skipped (empty if it was saved)."""
    raw = tifffile.imread(path)
    reasons = []
    if raw.ndim != len(ref_shape):
        reasons.append(f"ndim {raw.ndim} != {len(ref_shape)} (shape {raw.shape} vs {ref_shape})")
    elif raw.shape[CHANNEL_AXIS] != ref_shape[CHANNEL_AXIS]:
        reasons.append(f"{raw.shape[CHANNEL_AXIS]} channels != {ref_shape[CHANNEL_AXIS]} (shape {raw.shape} vs {ref_shape})")
    if raw.ndim > CHANNEL_AXIS and CHANNEL_INDEX >= raw.shape[CHANNEL_AXIS]:
        reasons.append(f"CHANNEL_INDEX {CHANNEL_INDEX} out of range for {raw.shape[CHANNEL_AXIS]} channels")
    if reasons:
        return reasons

    stack = np.moveaxis(np.take(raw, CHANNEL_INDEX, axis=CHANNEL_AXIS), Z_AXIS, 0)
    subtracted, _ = subtract_background_image(stack, rolling_ball_background(stack, ROLLING_BALL_RADIUS))
    sum_projection = subtracted.sum(axis=0).astype(np.float32)  # float32: a sum can overflow uint16
    tifffile.imwrite(out_dir / f"{path.stem}{SUFFIX}.tif", sum_projection)
    return []


out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

paths = sorted(Path(INPUT_DIR).glob(PATTERN))
ref_shape = tifffile.TiffFile(paths[0]).series[0].shape  # the others must match its ndim and channel count
todo = [p for p in paths if OVERWRITE or not (out_dir / f"{p.stem}{SUFFIX}.tif").exists()]
print(f"{len(paths)} files matching {PATTERN}, {len(paths) - len(todo)} already done, "
      f"{len(todo)} to process on {WORKERS} workers (reference shape {ref_shape})")

skipped = []  # (file name, reasons)
with ThreadPoolExecutor(WORKERS) as pool:  # scipy.ndimage releases the GIL, so threads run in parallel
    futures = {pool.submit(process, p, ref_shape): p for p in todo}
    for i, future in enumerate(as_completed(futures), 1):
        path = futures[future]
        try:
            reasons = future.result()
        except Exception as err:  # unreadable file etc.: log it and move on
            reasons = [f"{type(err).__name__}: {err}"]
        if reasons:
            skipped.append((path.name, reasons))
        print(f"[{i}/{len(todo)}] {'SKIP ' if reasons else ''}{path.name}"
              + (f": {'; '.join(reasons)}" if reasons else ""))

# --- Log skipped files ---------------------------------------------------------------
log_path = out_dir / f"skipped{SUFFIX}.log"
log_path.write_text("".join(f"{name}\t{'; '.join(reasons)}\n" for name, reasons in sorted(skipped)))
print(f"done: {len(todo) - len(skipped)} saved, {len(skipped)} skipped -> {log_path}")

585 files matching *.tif, 0 already done, 585 to process on 4 workers (reference shape (11, 4, 2048, 2048))
[1/585] 080623_CPEB4_Dox_INF-24hpi_001.tif
[2/585] 080623_CPEB4_Dox_INF-24hpi_002.tif
[3/585] 080623_CPEB4_Dox_INF-24hpi_004.tif
[4/585] 080623_CPEB4_Dox_INF-24hpi_003.tif
[5/585] 080623_CPEB4_Dox_INF-24hpi_005.tif
[6/585] 080623_CPEB4_Dox_INF-24hpi_007.tif
[7/585] 080623_CPEB4_Dox_INF-24hpi_008.tif
[8/585] 080623_CPEB4_Dox_INF-24hpi_006.tif
[9/585] 080623_CPEB4_Dox_INF-24hpi_009.tif
[10/585] 080623_CPEB4_Dox_INF-24hpi_010.tif
[11/585] 080623_CPEB4_Dox_INF-48hpi_001.tif
[12/585] 080623_CPEB4_Dox_INF-48hpi_002.tif
[13/585] 080623_CPEB4_Dox_INF-48hpi_003.tif
[14/585] 080623_CPEB4_Dox_INF-48hpi_004.tif
[15/585] 080623_CPEB4_Dox_INF-48hpi_005.tif
[16/585] 080623_CPEB4_Dox_INF-48hpi_006.tif
[17/585] 080623_CPEB4_Dox_INF-48hpi_007.tif
[18/585] 080623_CPEB4_Dox_INF-48hpi_008.tif
[19/585] 080623_CPEB4_Dox_INF-48hpi_009.tif
[20/585] 080623_CPEB4_Dox_INF-48hpi_010.tif
[21/585] 080623_CPEB4

## 2. Measure foci count and intensities per nucleus

Inputs (2D, same shape), matched by basename (the raw file stem) from three user-defined directories:
- **nucleus label image**: `{basename}{LABEL_SUFFIX}.tif` (cellpose output). The label value is the nucleus id (`nucleus-id`).
- **foci mask**: `{basename}{FOCI_SUFFIX}.tif`. Any non-zero pixel is foreground.
- **sum intensity image**: `{basename}{SUM_SUFFIX}.tif` (section 1 output).

Rules:
- **Border nuclei**: nuclei touching the image sides are discarded. The remaining ids are kept as they are in the label image, never renumbered, so they can be cross-referenced.
- **Foci**: 4-connected blobs of the foci mask (diagonal-only contact = separate foci), given an arbitrary per-image `foci-index` that is not meant for cross-referencing.
- **Assignment**: each whole focus goes to the nucleus it overlaps most. Any overlap counts, even one pixel. Ties go to the lowest nucleus id and are logged. The majority is decided among *all* nuclei, so a focus won by a discarded border nucleus is dropped, not handed to a neighbour.
- **Foci measurements** use the whole blob, including pixels outside the nucleus (flagged by `foci-extends-outside-nucleus`).
- Foci that overlap no nucleus are ignored.
- **Minimum foci area**: foci with area < `MIN_FOCI_AREA` pixels (default 4) are removed before assignment, so they are neither counted nor able to win a nucleus. No other size or intensity filters are applied: foci masks and nuclei are otherwise taken as segmented.
- Nuclei with zero foci are **kept** in the per-nucleus table (`foci-count` = 0, totals = 0, mean = NaN).
- Areas are in pixels. Intensities are sums of sum-projection pixel values.

Outputs (in `OUTPUT_DIR`):
1. `foci_intensity_quantification.csv`: one row per counted focus (nuclei with zero foci have no rows here). Nucleus columns repeat across the foci of a nucleus.
    - `image-filename-basename`, `nucleus-id`, `nucleus-area`, `nucleus-fish-integrated-intensity`, `foci-index`, `foci-area`, `foci-fish-integrated-intensity`, `foci-max-intensity`, `foci-centroid-x`, `foci-centroid-y` (pixels; x = column, y = row), `foci-extends-outside-nucleus`
2. `foci_intensity_quantification_summarised-per-nucleus.csv`: one row per kept nucleus, built from the label image (not from table 1), so nuclei with zero foci are included.
    - `image-filename-basename`, `nucleus-id`, `nucleus-area`, `nucleus-fish-integrated-intensity`, `foci-count`, `total-foci-area`, `total-foci-fish-integrated-intensity`, `mean-fish-integrated-intensity-per-foci` (= total / count)
3. `foci_intensity_quantification.log`: tab-separated `image  level  message`.
    - `ERROR`: missing or mismatched inputs, unreadable files.
    - `WARNING`: sum images with no label image; images with no nuclei or only border nuclei.
    - `OUTLIER`: foci overlapping more than one nucleus (with pixel counts and the winner).
    - `INFO`: per-image nucleus and foci counts (nuclei kept / border; foci counted / below the minimum area / on border nuclei / outside nuclei).
4. `foci_intensity_quantification_params.json`: all settings, the run time, file counts and package versions.
5. `qc/{basename}_nucleus-foci-qc.png`: sum projection at 1:1 pixels. Kept nuclei in cyan with their id, border nuclei in grey, counted foci in magenta, ignored foci in yellow (too small, on a border nucleus, or outside nuclei).

### Planned, not yet implemented
To be added **after** the Mock nuclear background has been subtracted downstream:
- mean intensity (integrated / area) for nuclei and foci
- nucleoplasm: the nucleus minus its foci (area, integrated and mean intensity)
- fraction of nuclear signal in foci = `total-foci-fish-integrated-intensity` / `nucleus-fish-integrated-intensity`. Note that foci are measured as whole blobs, so pixels outside the nucleus are included in the numerator. For this ratio, either clip foci to the nucleus or report both versions.


In [ ]:
import csv
import json
import math
import sys
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

sys.path.insert(0, "src")

import numpy as np
import tifffile

from nucleus_foci_quant import FOCI_COLUMNS, NUCLEUS_COLUMNS, measure_image, qc_figure

# --- User-defined -------------------------------------------------------------------
LABEL_DIR = "/Volumes/Jeff-exFAT/Marko_HIV/Marko_HIV_nucleuslabels"  # cellpose output
FOCI_DIR = "/Volumes/Jeff-exFAT/Marko_HIV/Marko_HIV_focimask"
SUM_DIR = "/Volumes/Jeff-exFAT/Marko_HIV/Marko_HIV_ch0bgs100sum"
OUTPUT_DIR = "/Volumes/Jeff-exFAT/Marko_HIV/Marko_HIV_foci-quantification"
LABEL_SUFFIX = "_cp_masks"   # {basename}{LABEL_SUFFIX}.tif
FOCI_SUFFIX = "_focimask"    # {basename}{FOCI_SUFFIX}.tif
SUM_SUFFIX = "_ch0bgs100sum" # {basename}{SUM_SUFFIX}.tif
MIN_FOCI_AREA = 6            # px; foci with area < this are removed (0 keeps all)
SAVE_QC = True               # one PNG per image in OUTPUT_DIR/qc
WORKERS = 4


# --- Measure one image ----------------------------------------------------------------
def process(basename):
    """Returns (foci_rows, nucleus_rows, messages). Raises on unreadable/missing inputs."""
    paths = {
        "label": Path(LABEL_DIR) / f"{basename}{LABEL_SUFFIX}.tif",
        "foci": Path(FOCI_DIR) / f"{basename}{FOCI_SUFFIX}.tif",
        "sum": Path(SUM_DIR) / f"{basename}{SUM_SUFFIX}.tif",
    }
    missing = [f"{kind} ({p.name})" for kind, p in paths.items() if not p.exists()]
    if missing:
        raise FileNotFoundError(f"missing {', '.join(missing)}")
    labels, foci, intensity = (tifffile.imread(paths[k]) for k in ("label", "foci", "sum"))
    foci_rows, nucleus_rows, messages, foci_lab, assigned = measure_image(
        basename, labels, foci, intensity, min_foci_area=MIN_FOCI_AREA)
    if SAVE_QC:
        qc_figure(basename, labels, intensity, foci_lab, assigned).savefig(
            qc_dir / f"{basename}_nucleus-foci-qc.png")
    return foci_rows, nucleus_rows, messages


def write_csv(path, columns, rows):
    with open(path, "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns)
        writer.writeheader()
        writer.writerows({k: ("" if isinstance(v, float) and math.isnan(v) else v) for k, v in r.items()}
                         for r in rows)


out_dir = Path(OUTPUT_DIR)
qc_dir = out_dir / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
started = datetime.now()

label_paths = sorted(Path(LABEL_DIR).glob(f"*{LABEL_SUFFIX}.tif"))
basenames = [p.name.removesuffix(f"{LABEL_SUFFIX}.tif") for p in label_paths]
log = []  # (basename, level, message)
for p in sorted(Path(SUM_DIR).glob(f"*{SUM_SUFFIX}.tif")):
    b = p.name.removesuffix(f"{SUM_SUFFIX}.tif")
    if b not in set(basenames):
        log.append((b, "WARNING", "sum intensity image has no nucleus label image"))
print(f"{len(basenames)} label images matching *{LABEL_SUFFIX}.tif, processing on {WORKERS} workers")

all_foci, all_nuclei, n_failed = [], [], 0
with ThreadPoolExecutor(WORKERS) as pool:
    futures = {pool.submit(process, b): b for b in basenames}
    for i, future in enumerate(as_completed(futures), 1):
        b = futures[future]
        try:
            foci_rows, nucleus_rows, messages = future.result()
        except Exception as err:  # log it and move on
            n_failed += 1
            log.append((b, "ERROR", f"{type(err).__name__}: {err}"))
            print(f"[{i}/{len(basenames)}] ERROR {b}: {err}")
            continue
        all_foci += foci_rows
        all_nuclei += nucleus_rows
        log += [(b, level, msg) for level, msg in messages]
        n_outliers = sum(level == "OUTLIER" for level, _ in messages)
        print(f"[{i}/{len(basenames)}] {b}: {len(nucleus_rows)} nuclei, {len(foci_rows)} foci"
              + (f", {n_outliers} outlier(s)" if n_outliers else ""))

# --- Save tables, log and parameters ----------------------------------------------------
all_foci.sort(key=lambda r: (r["image-filename-basename"], r["nucleus-id"], r["foci-index"]))
all_nuclei.sort(key=lambda r: (r["image-filename-basename"], r["nucleus-id"]))
write_csv(out_dir / "foci_intensity_quantification.csv", FOCI_COLUMNS, all_foci)
write_csv(out_dir / "foci_intensity_quantification_summarised-per-nucleus.csv", NUCLEUS_COLUMNS, all_nuclei)

log_path = out_dir / "foci_intensity_quantification.log"
with open(log_path, "w") as fh:
    fh.write("image\tlevel\tmessage\n")
    fh.writelines(f"{b}\t{level}\t{msg}\n" for b, level, msg in sorted(log))

params = {
    "started": started.isoformat(timespec="seconds"),
    "finished": datetime.now().isoformat(timespec="seconds"),
    "LABEL_DIR": LABEL_DIR, "FOCI_DIR": FOCI_DIR, "SUM_DIR": SUM_DIR, "OUTPUT_DIR": OUTPUT_DIR,
    "LABEL_SUFFIX": LABEL_SUFFIX, "FOCI_SUFFIX": FOCI_SUFFIX, "SUM_SUFFIX": SUM_SUFFIX,
    "MIN_FOCI_AREA": MIN_FOCI_AREA, "SAVE_QC": SAVE_QC, "WORKERS": WORKERS,
    "rules": {
        "foci_connectivity": 4,
        "border_nuclei": "excluded; ids kept from the label image",
        "foci_assignment": "whole blob to the max-overlap nucleus among all nuclei; any overlap counts; "
                           "ties to lowest id; dropped if the winner is a border nucleus",
        "foci_measurement": "whole blob, including pixels outside the nucleus",
        "size_filters": f"foci with area < {MIN_FOCI_AREA} px removed before assignment; no nucleus filters",
    },
    "images": {"label_images": len(basenames), "processed": len(basenames) - n_failed, "failed": n_failed},
    "rows": {"foci": len(all_foci), "nuclei": len(all_nuclei)},
    "versions": {"python": sys.version.split()[0],
                 **{pkg: version(pkg) for pkg in ("numpy", "scipy", "scikit-image", "tifffile", "matplotlib")}},
}
(out_dir / "foci_intensity_quantification_params.json").write_text(json.dumps(params, indent=2))

print(f"done: {len(basenames) - n_failed} images, {len(all_nuclei)} nuclei, {len(all_foci)} foci, "
      f"{n_failed} failed -> {out_dir}")
